In [1]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# 1. Chargement du dataset labélisé à l'Étape 4
df = pd.read_csv('../data/processed/df_classified_target.csv')

# Séparation des features (X) et de la cible (y)
X = df.drop(columns=['Cluster', 'risk_category'])
y = df['risk_category']

# 2. Séparation Train / Test (80% train, 20% test avec stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Définition des Pipelines (Scaler + SMOTE + Classifieur)
pipelines = {
    'LogisticRegression': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'XGBoost': ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))
    ])
}

# Grilles d'hyperparamètres pour GridSearchCV
params = {
    'LogisticRegression': {
        'model__C': [0.1, 1.0, 10.0]
    },
    'RandomForest': {
        'model__n_estimators': [50, 100],
        'model__max_depth': [3, 5, 10]
    },
    'XGBoost': {
        'model__n_estimators': [50, 100],
        'model__max_depth': [3, 5],
        'model__learning_rate': [0.01, 0.1]
    }
}

# 4. Entraînement et Optimisation des Modèles
best_models = {}
results = []

for name in pipelines:
    grid = GridSearchCV(pipelines[name], params[name], cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    y_pred = grid.predict(X_test)
    
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Précision': precision_score(y_test, y_pred),
        'Rappel': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'Best_Params': grid.best_params_
    })
    best_models[name] = grid.best_estimator_

# 5. Comparaison des Performances
df_results = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print("--- Comparatif des Modèles de Classification ---")
print(df_results[['Modèle', 'Accuracy', 'Précision', 'Rappel', 'F1-Score']])

# 6. Sélection et Évaluation du Meilleur Modèle
best_model_name = df_results.iloc[0]['Modèle']
final_pipeline = best_models[best_model_name]
print(f"\nMeilleur modèle sélectionné : {best_model_name}")

y_pred_best = final_pipeline.predict(X_test)

# Matrice de Confusion
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues')
plt.title(f'Matrice de Confusion - {best_model_name}')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

# Classification Report
print("\nRapport de Classification détaillé :")
print(classification_report(y_test, y_pred_best))

# 7. Sauvegarde du Pipeline complet pour l'API (Étape 7)
joblib.dump(final_pipeline, '../src/models/best_diabetes_pipeline.joblib')
print("✅ Pipeline sauvegardé sous 'src/models/best_diabetes_pipeline.joblib'.")

KeyError: "['Cluster', 'risk_category'] not found in axis"

In [5]:
!pip install joblib imbalanced-learn xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 734.9 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 862.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 65.0/131.7 MB 95.7 kB/s eta 0:11:38
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pip/_vendor/urllib3/response.py", line 438, in _error_catcher
    yield
  File "/usr/local/lib/python3.10/site-packages/pip/_vendor/urllib3/response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
  File "/usr/local/lib/python3.10/site-packages/pip/_vendor/urllib3/response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
  File "/usr/local/lib/python3.10/site-packages/pip/_vendor/cachecontrol/filewrapper.py", line 90, in read
    data = self.__fp.read(amt)
  File "/usr/local/lib/python3.10/http/client.py", line 481, in read
    s = self.fp.read(amt)
  F